# Advanced Problems with Solutions: JSON Serialization in Python

This notebook goes beyond basic `json.dumps()` / `json.loads()` usage and focuses on **real-world serialization design**.

## What you will practice

- JSON-compatible Python types and round-trip caveats
- strict JSON vs Python's permissive defaults
- `default=...` custom serialization
- `object_hook` and tagged-object deserialization
- `JSONEncoder` subclasses
- `Decimal`, `Fraction`, `complex`, `set`, `tuple`
- `datetime`, `date`, `time`, `UUID`, `Enum`, `bytes`
- dataclasses and nested domain models
- duplicate-key detection
- high-precision numeric parsing
- deterministic/canonical JSON
- schema versioning and migrations
- JSON Lines (`.jsonl`) streaming
- circular-reference failures
- validation and defensive decoding
- security and interoperability best practices

All exercises include a **complete solution** and executable checks.


## Best-practice checklist

1. Treat JSON as a **wire/storage format**, not as a perfect mirror of arbitrary Python objects.
2. Define an explicit representation for non-JSON-native types.
3. Prefer **tagged objects** when type restoration matters.
4. Validate data after deserialization; JSON parsing alone does not validate business rules.
5. Use `allow_nan=False` when producing interoperable strict JSON.
6. Use `Decimal` when exact decimal semantics matter.
7. Avoid silently converting dictionary keys unless the behavior is intentional.
8. Do not rely on JSON object key order for semantic meaning.
9. Prefer versioned payloads for long-lived formats.
10. For large datasets, stream records rather than loading everything into memory.
11. Never deserialize into arbitrary executable objects merely because the input contains a type name.
12. Keep encoders/decoders small, explicit, testable, and deterministic.


In [1]:

from __future__ import annotations

import base64
import dataclasses
import json
import math
from dataclasses import dataclass, asdict
from datetime import date, datetime, time, timezone
from decimal import Decimal
from enum import Enum
from fractions import Fraction
from pathlib import Path
from typing import Any
from uuid import UUID, uuid4


## Warm-up: JSON's native type mapping

Python's standard encoder naturally supports values that can be represented as:

- `dict` with compatible keys
- `list` / `tuple`
- `str`
- `int`
- `float`
- `True` / `False`
- `None`

But some conversions lose Python-specific type information.


In [2]:

sample = {
    "text": "hello",
    "integer": 42,
    "float": 3.5,
    "boolean": True,
    "nothing": None,
    "array": [1, 2, 3],
    "tuple_value": (10, 20),
}

encoded = json.dumps(sample)
decoded = json.loads(encoded)

print(encoded)
print(decoded)
print(type(decoded["tuple_value"]))   # list, not tuple


{"text": "hello", "integer": 42, "float": 3.5, "boolean": true, "nothing": null, "array": [1, 2, 3], "tuple_value": [10, 20]}
{'text': 'hello', 'integer': 42, 'float': 3.5, 'boolean': True, 'nothing': None, 'array': [1, 2, 3], 'tuple_value': [10, 20]}
<class 'list'>


# Problem 1 — Diagnose round-trip failures

Consider the object below.

Tasks:

1. Serialize it with `json.dumps`.
2. Deserialize it with `json.loads`.
3. Determine why equality fails.
4. Write a corrected representation that round-trips predictably.


In [3]:

original = {
    1: "integer key",
    2: "another integer key",
    "coords": (10, 20),
}


## Solution 1


In [4]:

serialized = json.dumps(original)
restored = json.loads(serialized)

print("JSON:", serialized)
print("Restored:", restored)
print("Equal?", original == restored)

# Why equality fails:
# 1) JSON object keys are strings, so integer keys become strings.
# 2) tuples become JSON arrays and come back as Python lists.


JSON: {"1": "integer key", "2": "another integer key", "coords": [10, 20]}
Restored: {'1': 'integer key', '2': 'another integer key', 'coords': [10, 20]}
Equal? False


In [5]:

# Best practice: choose an explicit JSON-native representation.

normalized = {
    "items": [
        {"key": 1, "value": "integer key"},
        {"key": 2, "value": "another integer key"},
    ],
    "coords": [10, 20],
}

payload = json.dumps(normalized)
round_trip = json.loads(payload)

assert round_trip == normalized
round_trip


{'items': [{'key': 1, 'value': 'integer key'},
  {'key': 2, 'value': 'another integer key'}],
 'coords': [10, 20]}

# Problem 2 — Reject non-standard NaN and Infinity

Python's `json` module is permissive by default and may emit `NaN`, `Infinity`, and `-Infinity`.
Those tokens are not valid in strict JSON.

Tasks:

1. Demonstrate the permissive behavior.
2. Make serialization fail for non-finite floats.
3. Write a validator that recursively detects non-finite numbers before serialization.


## Solution 2


In [6]:

numbers = {
    "ok": 1.25,
    "nan": float("nan"),
    "pos_inf": float("inf"),
    "neg_inf": float("-inf"),
}

print(json.dumps(numbers))


{"ok": 1.25, "nan": NaN, "pos_inf": Infinity, "neg_inf": -Infinity}


In [7]:

try:
    json.dumps(numbers, allow_nan=False)
except ValueError as exc:
    print("Strict mode rejected payload:", exc)


Strict mode rejected payload: Out of range float values are not JSON compliant: nan


In [8]:

def assert_finite_numbers(value: Any, path: str = "$") -> None:
    if isinstance(value, float):
        if not math.isfinite(value):
            raise ValueError(f"Non-finite float at {path}: {value!r}")
    elif isinstance(value, dict):
        for key, item in value.items():
            assert_finite_numbers(item, f"{path}.{key}")
    elif isinstance(value, (list, tuple)):
        for index, item in enumerate(value):
            assert_finite_numbers(item, f"{path}[{index}]")

try:
    assert_finite_numbers(numbers)
except ValueError as exc:
    print(exc)


Non-finite float at $.nan: nan


# Problem 3 — Serialize `Decimal` without losing precision

You receive financial values as `Decimal`.

Tasks:

1. Show that `Decimal` is not natively serializable.
2. Explain why converting to `float` can be dangerous.
3. Serialize `Decimal` as a tagged object.
4. Restore it back to `Decimal`.


## Solution 3


In [9]:

amount = Decimal("1234567890.12345678901234567890")

try:
    json.dumps({"amount": amount})
except TypeError as exc:
    print(exc)


Object of type Decimal is not JSON serializable


In [10]:

as_float = float(amount)
print("Decimal:", amount)
print("Float:  ", as_float)
print("Back to Decimal from float string:", Decimal(str(as_float)))


Decimal: 1234567890.12345678901234567890
Float:   1234567890.1234567
Back to Decimal from float string: 1234567890.1234567


In [11]:

def encode_decimal(obj: Any) -> Any:
    if isinstance(obj, Decimal):
        return {
            "__type__": "decimal",
            "value": str(obj),
        }
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def decode_tagged_object(obj: dict[str, Any]) -> Any:
    if obj.get("__type__") == "decimal":
        return Decimal(obj["value"])
    return obj


payload = json.dumps({"amount": amount}, default=encode_decimal)
restored = json.loads(payload, object_hook=decode_tagged_object)

print(payload)
print(restored)
print(type(restored["amount"]))
assert restored["amount"] == amount


{"amount": {"__type__": "decimal", "value": "1234567890.12345678901234567890"}}
{'amount': Decimal('1234567890.12345678901234567890')}
<class 'decimal.Decimal'>


# Problem 4 — Build a reusable multi-type `default` serializer

Create one serializer that supports:

- `Decimal`
- `Fraction`
- `complex`
- `set`
- `tuple` (discussion caveat)
- `datetime`
- `date`
- `time`
- `UUID`
- `bytes`

Use explicit tagged objects wherever Python type restoration matters.


## Solution 4


In [12]:

def json_default(obj: Any) -> Any:
    if isinstance(obj, Decimal):
        return {"__type__": "decimal", "value": str(obj)}

    if isinstance(obj, Fraction):
        return {
            "__type__": "fraction",
            "numerator": obj.numerator,
            "denominator": obj.denominator,
        }

    if isinstance(obj, complex):
        return {
            "__type__": "complex",
            "real": obj.real,
            "imag": obj.imag,
        }

    if isinstance(obj, set):
        # Sorting gives deterministic output when values are comparable.
        try:
            values = sorted(obj)
        except TypeError:
            values = list(obj)
        return {"__type__": "set", "items": values}

    # Important:
    # Standard json sees tuples as sequences before `default` is invoked,
    # so this branch will normally NOT be reached for plain tuples.
    if isinstance(obj, tuple):
        return {"__type__": "tuple", "items": list(obj)}

    if isinstance(obj, datetime):
        return {"__type__": "datetime", "value": obj.isoformat()}

    # datetime is also a date, so datetime must be checked first.
    if isinstance(obj, date):
        return {"__type__": "date", "value": obj.isoformat()}

    if isinstance(obj, time):
        return {"__type__": "time", "value": obj.isoformat()}

    if isinstance(obj, UUID):
        return {"__type__": "uuid", "value": str(obj)}

    if isinstance(obj, bytes):
        return {
            "__type__": "bytes",
            "encoding": "base64",
            "value": base64.b64encode(obj).decode("ascii"),
        }

    raise TypeError(f"Unsupported type: {type(obj).__name__}")


In [13]:

example = {
    "price": Decimal("19.99"),
    "ratio": Fraction(2, 7),
    "signal": 3 + 4j,
    "tags": {"python", "json", "serialization"},
    "created_at": datetime(2026, 8, 7, 12, 30, tzinfo=timezone.utc),
    "birthday": date(1990, 1, 2),
    "alarm": time(7, 45, 30),
    "id": uuid4(),
    "blob": b"\x00\x01hello\xff",
}

payload = json.dumps(example, default=json_default, indent=2, sort_keys=True)
print(payload)


{
  "alarm": {
    "__type__": "time",
    "value": "07:45:30"
  },
  "birthday": {
    "__type__": "date",
    "value": "1990-01-02"
  },
  "blob": {
    "__type__": "bytes",
    "encoding": "base64",
    "value": "AAFoZWxsb/8="
  },
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T12:30:00+00:00"
  },
  "id": {
    "__type__": "uuid",
    "value": "f569def7-bd00-448d-91e2-1f865d9341de"
  },
  "price": {
    "__type__": "decimal",
    "value": "19.99"
  },
  "ratio": {
    "__type__": "fraction",
    "denominator": 7,
    "numerator": 2
  },
  "signal": {
    "__type__": "complex",
    "imag": 4.0,
    "real": 3.0
  },
  "tags": {
    "__type__": "set",
    "items": [
      "json",
      "python",
      "serialization"
    ]
  }
}


# Problem 5 — Complete the tagged-object decoder

Write an `object_hook` that reverses the tagged representations from Problem 4.
Unknown objects must remain ordinary dictionaries.


## Solution 5


In [14]:

def tagged_object_hook(obj: dict[str, Any]) -> Any:
    type_name = obj.get("__type__")

    if type_name == "decimal":
        return Decimal(obj["value"])

    if type_name == "fraction":
        return Fraction(obj["numerator"], obj["denominator"])

    if type_name == "complex":
        return complex(obj["real"], obj["imag"])

    if type_name == "set":
        return set(obj["items"])

    if type_name == "tuple":
        return tuple(obj["items"])

    if type_name == "datetime":
        return datetime.fromisoformat(obj["value"])

    if type_name == "date":
        return date.fromisoformat(obj["value"])

    if type_name == "time":
        return time.fromisoformat(obj["value"])

    if type_name == "uuid":
        return UUID(obj["value"])

    if type_name == "bytes":
        if obj.get("encoding") != "base64":
            raise ValueError("Unsupported bytes encoding")
        return base64.b64decode(obj["value"].encode("ascii"))

    return obj


In [15]:

restored = json.loads(payload, object_hook=tagged_object_hook)

for key, value in restored.items():
    print(f"{key:12} -> {value!r:55} {type(value).__name__}")


alarm        -> datetime.time(7, 45, 30)                                time
birthday     -> datetime.date(1990, 1, 2)                               date
blob         -> b'\x00\x01hello\xff'                                    bytes
created_at   -> datetime.datetime(2026, 8, 7, 12, 30, tzinfo=datetime.timezone.utc) datetime
id           -> UUID('f569def7-bd00-448d-91e2-1f865d9341de')            UUID
price        -> Decimal('19.99')                                        Decimal
ratio        -> Fraction(2, 7)                                          Fraction
signal       -> (3+4j)                                                  complex
tags         -> {'json', 'python', 'serialization'}                     set


In [16]:

assert restored["price"] == example["price"]
assert restored["ratio"] == example["ratio"]
assert restored["signal"] == example["signal"]
assert restored["tags"] == example["tags"]
assert restored["created_at"] == example["created_at"]
assert restored["birthday"] == example["birthday"]
assert restored["alarm"] == example["alarm"]
assert restored["id"] == example["id"]
assert restored["blob"] == example["blob"]
print("All supported tagged values round-tripped successfully.")


All supported tagged values round-tripped successfully.


# Problem 6 — Preserve tuples, including nested tuples

A subtle issue: the standard encoder converts tuples to JSON arrays **before** calling `default`.

Therefore this will not preserve tuples automatically:

```python
json.dumps({"point": (1, 2)}, default=json_default)
```

Design a recursive **preprocessing** function that converts Python values into an explicitly tagged JSON-compatible tree before calling `json.dumps`.


## Solution 6


In [17]:

def to_json_tree(value: Any) -> Any:
    # Preserve distinction between list and tuple.
    if isinstance(value, tuple):
        return {
            "__type__": "tuple",
            "items": [to_json_tree(item) for item in value],
        }

    if isinstance(value, list):
        return [to_json_tree(item) for item in value]

    if isinstance(value, dict):
        result = {}
        for key, item in value.items():
            if not isinstance(key, str):
                raise TypeError(
                    f"JSON object keys must be strings; got {type(key).__name__}: {key!r}"
                )
            result[key] = to_json_tree(item)
        return result

    if isinstance(value, set):
        items = [to_json_tree(item) for item in value]
        return {"__type__": "set", "items": items}

    if isinstance(value, Decimal):
        return {"__type__": "decimal", "value": str(value)}

    if isinstance(value, Fraction):
        return {
            "__type__": "fraction",
            "numerator": value.numerator,
            "denominator": value.denominator,
        }

    if isinstance(value, complex):
        return {
            "__type__": "complex",
            "real": value.real,
            "imag": value.imag,
        }

    if isinstance(value, datetime):
        return {"__type__": "datetime", "value": value.isoformat()}

    if isinstance(value, date):
        return {"__type__": "date", "value": value.isoformat()}

    if isinstance(value, time):
        return {"__type__": "time", "value": value.isoformat()}

    if isinstance(value, UUID):
        return {"__type__": "uuid", "value": str(value)}

    if isinstance(value, bytes):
        return {
            "__type__": "bytes",
            "encoding": "base64",
            "value": base64.b64encode(value).decode("ascii"),
        }

    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    raise TypeError(f"Unsupported type: {type(value).__name__}")


In [18]:

nested = {
    "point": (10, 20),
    "matrix_row": [(1, 2), (3, 4)],
    "deep": {"x": (Decimal("1.25"), (7, 8))},
}

tree = to_json_tree(nested)
payload = json.dumps(tree, indent=2)
restored = json.loads(payload, object_hook=tagged_object_hook)

print(payload)
print(restored)

assert restored["point"] == (10, 20)
assert restored["matrix_row"][0] == (1, 2)
assert restored["deep"]["x"][1] == (7, 8)


{
  "point": {
    "__type__": "tuple",
    "items": [
      10,
      20
    ]
  },
  "matrix_row": [
    {
      "__type__": "tuple",
      "items": [
        1,
        2
      ]
    },
    {
      "__type__": "tuple",
      "items": [
        3,
        4
      ]
    }
  ],
  "deep": {
    "x": {
      "__type__": "tuple",
      "items": [
        {
          "__type__": "decimal",
          "value": "1.25"
        },
        {
          "__type__": "tuple",
          "items": [
            7,
            8
          ]
        }
      ]
    }
  }
}
{'point': (10, 20), 'matrix_row': [(1, 2), (3, 4)], 'deep': {'x': (Decimal('1.25'), (7, 8))}}


# Problem 7 — Custom `JSONEncoder` subclass

Some codebases prefer a custom encoder class.

Tasks:

1. Create an encoder supporting `Decimal`, `Fraction`, `complex`, `set`, `datetime`, `UUID`, and `bytes`.
2. Call `super().default(obj)` for unsupported types.
3. Demonstrate serialization with `cls=AdvancedJSONEncoder`.


## Solution 7


In [19]:

class AdvancedJSONEncoder(json.JSONEncoder):
    def default(self, obj: Any) -> Any:
        if isinstance(obj, Decimal):
            return {"__type__": "decimal", "value": str(obj)}

        if isinstance(obj, Fraction):
            return {
                "__type__": "fraction",
                "numerator": obj.numerator,
                "denominator": obj.denominator,
            }

        if isinstance(obj, complex):
            return {
                "__type__": "complex",
                "real": obj.real,
                "imag": obj.imag,
            }

        if isinstance(obj, set):
            return {"__type__": "set", "items": sorted(obj)}

        if isinstance(obj, datetime):
            return {"__type__": "datetime", "value": obj.isoformat()}

        if isinstance(obj, UUID):
            return {"__type__": "uuid", "value": str(obj)}

        if isinstance(obj, bytes):
            return {
                "__type__": "bytes",
                "encoding": "base64",
                "value": base64.b64encode(obj).decode("ascii"),
            }

        return super().default(obj)


In [20]:

data = {
    "amount": Decimal("5.50"),
    "ratio": Fraction(7, 9),
    "z": 2 - 3j,
    "labels": {"a", "b", "c"},
    "ts": datetime(2026, 8, 7, tzinfo=timezone.utc),
    "id": uuid4(),
    "payload": b"abc",
}

encoded = json.dumps(data, cls=AdvancedJSONEncoder, indent=2)
decoded = json.loads(encoded, object_hook=tagged_object_hook)

print(encoded)
assert decoded["amount"] == Decimal("5.50")
assert decoded["ratio"] == Fraction(7, 9)


{
  "amount": {
    "__type__": "decimal",
    "value": "5.50"
  },
  "ratio": {
    "__type__": "fraction",
    "numerator": 7,
    "denominator": 9
  },
  "z": {
    "__type__": "complex",
    "real": 2.0,
    "imag": -3.0
  },
  "labels": {
    "__type__": "set",
    "items": [
      "a",
      "b",
      "c"
    ]
  },
  "ts": {
    "__type__": "datetime",
    "value": "2026-08-07T00:00:00+00:00"
  },
  "id": {
    "__type__": "uuid",
    "value": "e3360583-ab6e-4c87-a38d-28b15526391f"
  },
  "payload": {
    "__type__": "bytes",
    "encoding": "base64",
    "value": "YWJj"
  }
}


# Problem 8 — Dataclasses and nested domain models

Build JSON serialization for a small order domain.

Requirements:

- `OrderItem` and `Order` are dataclasses.
- Prices use `Decimal`.
- IDs use `UUID`.
- timestamps use timezone-aware `datetime`.
- payload must include a schema version.
- decoding should reconstruct the dataclasses.


## Solution 8


In [21]:

@dataclass(frozen=True)
class OrderItem:
    sku: str
    quantity: int
    unit_price: Decimal


@dataclass(frozen=True)
class Order:
    order_id: UUID
    customer: str
    items: list[OrderItem]
    created_at: datetime


In [22]:

def order_to_document(order: Order) -> dict[str, Any]:
    return {
        "schema_version": 1,
        "order": {
            "order_id": str(order.order_id),
            "customer": order.customer,
            "created_at": order.created_at.isoformat(),
            "items": [
                {
                    "sku": item.sku,
                    "quantity": item.quantity,
                    "unit_price": str(item.unit_price),
                }
                for item in order.items
            ],
        },
    }


def order_from_document(document: dict[str, Any]) -> Order:
    if document.get("schema_version") != 1:
        raise ValueError("Unsupported schema_version")

    raw = document["order"]

    items = [
        OrderItem(
            sku=item["sku"],
            quantity=int(item["quantity"]),
            unit_price=Decimal(item["unit_price"]),
        )
        for item in raw["items"]
    ]

    return Order(
        order_id=UUID(raw["order_id"]),
        customer=raw["customer"],
        items=items,
        created_at=datetime.fromisoformat(raw["created_at"]),
    )


In [23]:

order = Order(
    order_id=uuid4(),
    customer="Ada Lovelace",
    items=[
        OrderItem("KB-001", 2, Decimal("49.95")),
        OrderItem("MS-010", 1, Decimal("19.99")),
    ],
    created_at=datetime(2026, 8, 7, 14, 30, tzinfo=timezone.utc),
)

document = order_to_document(order)
payload = json.dumps(document, indent=2, sort_keys=True)
restored_order = order_from_document(json.loads(payload))

print(payload)
print(restored_order)

assert restored_order == order


{
  "order": {
    "created_at": "2026-08-07T14:30:00+00:00",
    "customer": "Ada Lovelace",
    "items": [
      {
        "quantity": 2,
        "sku": "KB-001",
        "unit_price": "49.95"
      },
      {
        "quantity": 1,
        "sku": "MS-010",
        "unit_price": "19.99"
      }
    ],
    "order_id": "39aa56e6-64df-4c2b-87a0-236abb68543d"
  },
  "schema_version": 1
}
Order(order_id=UUID('39aa56e6-64df-4c2b-87a0-236abb68543d'), customer='Ada Lovelace', items=[OrderItem(sku='KB-001', quantity=2, unit_price=Decimal('49.95')), OrderItem(sku='MS-010', quantity=1, unit_price=Decimal('19.99'))], created_at=datetime.datetime(2026, 8, 7, 14, 30, tzinfo=datetime.timezone.utc))


# Problem 9 — Validate decoded JSON defensively

Parsing JSON only proves the text has valid JSON syntax. It does **not** prove that fields have the correct meaning.

Add validation rules:

- `customer` must be a non-empty string
- at least one item is required
- quantity must be a positive integer
- unit price must be non-negative
- `created_at` must include timezone information


## Solution 9


In [24]:

def require(condition: bool, message: str) -> None:
    if not condition:
        raise ValueError(message)


def validated_order_from_document(document: dict[str, Any]) -> Order:
    require(document.get("schema_version") == 1, "Unsupported schema_version")

    raw = document.get("order")
    require(isinstance(raw, dict), "'order' must be an object")

    customer = raw.get("customer")
    require(isinstance(customer, str) and customer.strip() != "",
            "'customer' must be a non-empty string")

    raw_items = raw.get("items")
    require(isinstance(raw_items, list) and len(raw_items) > 0,
            "'items' must be a non-empty array")

    created_at = datetime.fromisoformat(raw["created_at"])
    require(created_at.tzinfo is not None,
            "'created_at' must include timezone information")

    items: list[OrderItem] = []

    for index, raw_item in enumerate(raw_items):
        require(isinstance(raw_item, dict),
                f"items[{index}] must be an object")

        quantity = raw_item.get("quantity")
        require(isinstance(quantity, int) and not isinstance(quantity, bool) and quantity > 0,
                f"items[{index}].quantity must be a positive integer")

        price = Decimal(raw_item["unit_price"])
        require(price >= 0,
                f"items[{index}].unit_price must be non-negative")

        items.append(
            OrderItem(
                sku=str(raw_item["sku"]),
                quantity=quantity,
                unit_price=price,
            )
        )

    return Order(
        order_id=UUID(raw["order_id"]),
        customer=customer,
        items=items,
        created_at=created_at,
    )


In [25]:

bad_document = order_to_document(order)
bad_document["order"]["items"][0]["quantity"] = 0

try:
    validated_order_from_document(bad_document)
except ValueError as exc:
    print("Validation error:", exc)


Validation error: items[0].quantity must be a positive integer


# Problem 10 — Parse JSON floats directly as `Decimal`

Suppose an external API sends:

```json
{"price": 0.1, "tax": 0.2}
```

By default, JSON decimal numbers become Python `float`.

Tasks:

1. Parse normally.
2. Parse using `parse_float=Decimal`.
3. Compare the types and exact arithmetic.


## Solution 10


In [26]:

api_json = '{"price": 0.1, "tax": 0.2}'

normal = json.loads(api_json)
precise = json.loads(api_json, parse_float=Decimal)

print(normal, {k: type(v).__name__ for k, v in normal.items()})
print(precise, {k: type(v).__name__ for k, v in precise.items()})

print("float sum:  ", normal["price"] + normal["tax"])
print("Decimal sum:", precise["price"] + precise["tax"])

assert precise["price"] + precise["tax"] == Decimal("0.3")


{'price': 0.1, 'tax': 0.2} {'price': 'float', 'tax': 'float'}
{'price': Decimal('0.1'), 'tax': Decimal('0.2')} {'price': 'Decimal', 'tax': 'Decimal'}
float sum:   0.30000000000000004
Decimal sum: 0.3


## Extra numeric parsing example: customize integers too


In [27]:

numbers_json = '{"count": 12345678901234567890, "ratio": 1.25}'

custom = json.loads(
    numbers_json,
    parse_int=Decimal,
    parse_float=Decimal,
)

print(custom)
print(type(custom["count"]), type(custom["ratio"]))


{'count': Decimal('12345678901234567890'), 'ratio': Decimal('1.25')}
<class 'decimal.Decimal'> <class 'decimal.Decimal'>


# Problem 11 — Detect duplicate JSON object keys

JSON text can technically contain duplicate names:

```json
{"role": "user", "role": "admin"}
```

A normal `json.loads` call keeps only the last value.

For configuration/security-sensitive data, write a decoder that rejects duplicates using `object_pairs_hook`.


## Solution 11


In [28]:

duplicate_json = '{"role": "user", "role": "admin"}'

print("Default behavior:", json.loads(duplicate_json))


Default behavior: {'role': 'admin'}


In [29]:

def reject_duplicate_keys(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}

    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key!r}")
        result[key] = value

    return result


try:
    json.loads(duplicate_json, object_pairs_hook=reject_duplicate_keys)
except ValueError as exc:
    print("Rejected:", exc)


Rejected: Duplicate JSON key: 'role'


# Problem 12 — Deterministic / canonical-style JSON

You want logically equivalent dictionaries to produce the same byte sequence for hashing, caching, or signatures.

Tasks:

1. Use sorted keys.
2. Remove insignificant whitespace.
3. Reject NaN/Infinity.
4. Keep Unicode directly rather than escaping all non-ASCII characters.


## Solution 12


In [30]:

def deterministic_json(value: Any) -> str:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    )


a = {"z": 3, "a": 1, "message": "Здравей"}
b = {"message": "Здравей", "a": 1, "z": 3}

json_a = deterministic_json(a)
json_b = deterministic_json(b)

print(json_a)
print(json_b)
assert json_a == json_b


{"a":1,"message":"Здравей","z":3}
{"a":1,"message":"Здравей","z":3}


### Important note

`sort_keys=True` + compact separators gives **deterministic output for many ordinary cases**, but it is not automatically a complete implementation of every formal canonical-JSON specification.

If cryptographic interoperability depends on a specific canonicalization standard, implement that exact standard.


# Problem 13 — Schema versioning and migration

A long-lived application stored version-1 customer documents:

```json
{
  "schema_version": 1,
  "name": "Grace Hopper",
  "email": "grace@example.com"
}
```

Version 2 changes the format:

```json
{
  "schema_version": 2,
  "profile": {
    "full_name": "Grace Hopper",
    "emails": ["grace@example.com"]
  }
}
```

Write a migration function that accepts version 1 or version 2 and always returns version 2.


## Solution 13


In [31]:

def migrate_customer_document(document: dict[str, Any]) -> dict[str, Any]:
    version = document.get("schema_version")

    if version == 2:
        return document

    if version == 1:
        name = document["name"]
        email = document["email"]

        return {
            "schema_version": 2,
            "profile": {
                "full_name": name,
                "emails": [email] if email else [],
            },
        }

    raise ValueError(f"Unsupported schema version: {version!r}")


In [32]:

v1 = {
    "schema_version": 1,
    "name": "Grace Hopper",
    "email": "grace@example.com",
}

v2 = migrate_customer_document(v1)

print(json.dumps(v2, indent=2))
assert v2["schema_version"] == 2
assert v2["profile"]["emails"] == ["grace@example.com"]


{
  "schema_version": 2,
  "profile": {
    "full_name": "Grace Hopper",
    "emails": [
      "grace@example.com"
    ]
  }
}


# Problem 14 — Stream large data with JSON Lines

A single giant JSON array requires the full array to be built and often loaded as one document.

For logs/events, JSON Lines is frequently a better fit:

```text
{"event_id": 1, ...}
{"event_id": 2, ...}
{"event_id": 3, ...}
```

Tasks:

1. Write events one record per line.
2. Read them lazily line-by-line.
3. Skip blank lines.
4. Report malformed line numbers cleanly.


## Solution 14


In [33]:

events = [
    {"event_id": 1, "kind": "login", "user": "alice"},
    {"event_id": 2, "kind": "purchase", "user": "bob", "amount": "12.50"},
    {"event_id": 3, "kind": "logout", "user": "alice"},
]

jsonl_path = Path("events.jsonl")

with jsonl_path.open("w", encoding="utf-8") as f:
    for event in events:
        json.dump(event, f, ensure_ascii=False, allow_nan=False)
        f.write("\n")

print(jsonl_path.read_text(encoding="utf-8"))


{"event_id": 1, "kind": "login", "user": "alice"}
{"event_id": 2, "kind": "purchase", "user": "bob", "amount": "12.50"}
{"event_id": 3, "kind": "logout", "user": "alice"}



In [34]:

def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            stripped = line.strip()

            if not stripped:
                continue

            try:
                yield json.loads(stripped)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON on line {line_number}: {exc.msg}"
                ) from exc


for event in iter_jsonl(jsonl_path):
    print(event)


{'event_id': 1, 'kind': 'login', 'user': 'alice'}
{'event_id': 2, 'kind': 'purchase', 'user': 'bob', 'amount': '12.50'}
{'event_id': 3, 'kind': 'logout', 'user': 'alice'}


## Malformed JSON Lines example


In [35]:

bad_jsonl_path = Path("bad_events.jsonl")
bad_jsonl_path.write_text(
    '{"event_id": 1}\n'
    '{"event_id": 2\n'
    '{"event_id": 3}\n',
    encoding="utf-8",
)

try:
    list(iter_jsonl(bad_jsonl_path))
except ValueError as exc:
    print(exc)


Invalid JSON on line 2: Expecting ',' delimiter


# Problem 15 — Handle circular references

JSON represents trees, not arbitrary cyclic object graphs.

Create a cyclic Python structure and observe the failure.
Then design a JSON representation using object IDs instead of direct cyclic nesting.


## Solution 15


In [36]:

a = {"name": "A"}
b = {"name": "B", "parent": a}
a["child"] = b

try:
    json.dumps(a)
except ValueError as exc:
    print("Circular-reference error:", exc)


Circular-reference error: Circular reference detected


In [37]:

# Represent graph relationships explicitly using IDs.

graph_document = {
    "root_id": "A",
    "nodes": {
        "A": {
            "name": "A",
            "child_id": "B",
        },
        "B": {
            "name": "B",
            "parent_id": "A",
        },
    },
}

print(json.dumps(graph_document, indent=2))


{
  "root_id": "A",
  "nodes": {
    "A": {
      "name": "A",
      "child_id": "B"
    },
    "B": {
      "name": "B",
      "parent_id": "A"
    }
  }
}


In [38]:

def build_graph(document: dict[str, Any]) -> dict[str, dict[str, Any]]:
    raw_nodes = document["nodes"]

    nodes = {
        node_id: {"name": raw["name"]}
        for node_id, raw in raw_nodes.items()
    }

    for node_id, raw in raw_nodes.items():
        if "child_id" in raw:
            nodes[node_id]["child"] = nodes[raw["child_id"]]

        if "parent_id" in raw:
            nodes[node_id]["parent"] = nodes[raw["parent_id"]]

    return nodes


nodes = build_graph(graph_document)

assert nodes["A"]["child"] is nodes["B"]
assert nodes["B"]["parent"] is nodes["A"]
print(nodes["A"]["name"], "->", nodes["A"]["child"]["name"])


A -> B


# Problem 16 — Serialize Enums safely

Enums are common in domain models.

Create:

```python
class Status(Enum):
    PENDING = "pending"
    PAID = "paid"
    CANCELLED = "cancelled"
```

Design a representation that restores the enum and rejects unknown values.


## Solution 16


In [39]:

class Status(Enum):
    PENDING = "pending"
    PAID = "paid"
    CANCELLED = "cancelled"


In [40]:

def encode_status(status: Status) -> dict[str, str]:
    return {
        "__type__": "status",
        "value": status.value,
    }


def decode_status_object(obj: dict[str, Any]) -> Any:
    if obj.get("__type__") == "status":
        return Status(obj["value"])
    return obj


payload = json.dumps({"status": encode_status(Status.PAID)})
restored = json.loads(payload, object_hook=decode_status_object)

print(payload)
print(restored)
assert restored["status"] is Status.PAID


{"status": {"__type__": "status", "value": "paid"}}
{'status': <Status.PAID: 'paid'>}


In [41]:

bad_status_json = '{"status": {"__type__": "status", "value": "refunded"}}'

try:
    json.loads(bad_status_json, object_hook=decode_status_object)
except ValueError as exc:
    print("Unknown enum value rejected:", exc)


Unknown enum value rejected: 'refunded' is not a valid Status


# Problem 17 — One robust codec for a domain model

Build a small reusable codec abstraction.

Requirements:

- `dumps(value)` should produce strict deterministic JSON.
- `loads(text)` should reject duplicate keys.
- domain values are converted through explicit `to_document` / `from_document` functions.
- no arbitrary class imports or execution based on untrusted `"__type__"` values.


## Solution 17


In [42]:

class OrderJSONCodec:
    @staticmethod
    def dumps(order: Order) -> str:
        document = order_to_document(order)
        return json.dumps(
            document,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        )

    @staticmethod
    def loads(text: str) -> Order:
        document = json.loads(
            text,
            object_pairs_hook=reject_duplicate_keys,
        )
        return validated_order_from_document(document)


In [43]:

codec_payload = OrderJSONCodec.dumps(order)
codec_restored = OrderJSONCodec.loads(codec_payload)

print(codec_payload)
assert codec_restored == order


{"order":{"created_at":"2026-08-07T14:30:00+00:00","customer":"Ada Lovelace","items":[{"quantity":2,"sku":"KB-001","unit_price":"49.95"},{"quantity":1,"sku":"MS-010","unit_price":"19.99"}],"order_id":"39aa56e6-64df-4c2b-87a0-236abb68543d"},"schema_version":1}


# Problem 18 — Design a safe generic tagged decoder

A dangerous pattern is:

```python
# DO NOT DO THIS
module_name = obj["module"]
class_name = obj["class"]
# dynamically import and instantiate arbitrary classes...
```

Instead, implement an allowlist-based decoder registry.

Only registered tag names may be decoded.


## Solution 18


In [44]:

def decode_decimal_tag(obj: dict[str, Any]) -> Decimal:
    return Decimal(obj["value"])


def decode_fraction_tag(obj: dict[str, Any]) -> Fraction:
    return Fraction(obj["numerator"], obj["denominator"])


def decode_uuid_tag(obj: dict[str, Any]) -> UUID:
    return UUID(obj["value"])


SAFE_DECODERS = {
    "decimal": decode_decimal_tag,
    "fraction": decode_fraction_tag,
    "uuid": decode_uuid_tag,
}


def safe_registry_hook(obj: dict[str, Any]) -> Any:
    tag = obj.get("__type__")

    if tag is None:
        return obj

    decoder = SAFE_DECODERS.get(tag)

    if decoder is None:
        # Preserve unknown tagged data rather than executing anything.
        return obj

    return decoder(obj)


In [45]:

safe_json = json.dumps(
    {
        "money": {"__type__": "decimal", "value": "12.34"},
        "id": {"__type__": "uuid", "value": str(uuid4())},
        "unknown": {"__type__": "dangerous_class", "payload": "do not execute"},
    }
)

safe_result = json.loads(safe_json, object_hook=safe_registry_hook)

print(safe_result)
print(type(safe_result["money"]))
print(type(safe_result["id"]))
print(type(safe_result["unknown"]))


{'money': Decimal('12.34'), 'id': UUID('89b04ed5-5886-4b61-b7b1-84b92d635f15'), 'unknown': {'__type__': 'dangerous_class', 'payload': 'do not execute'}}
<class 'decimal.Decimal'>
<class 'uuid.UUID'>
<class 'dict'>


# Problem 19 — File I/O: `dump/load` vs `dumps/loads`

Create utility functions that save and load JSON configuration files correctly.

Requirements:

- UTF-8 encoding
- human-readable indentation
- strict NaN handling
- atomic-ish write using a temporary file followed by replacement
- clear JSON parsing errors


## Solution 19


In [46]:

def save_json_file(path: Path, data: Any) -> None:
    temp_path = path.with_suffix(path.suffix + ".tmp")

    with temp_path.open("w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        f.write("\n")

    temp_path.replace(path)


def load_json_file(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path} at line {exc.lineno}, "
            f"column {exc.colno}: {exc.msg}"
        ) from exc


In [47]:

config_path = Path("config.json")

config = {
    "app": "advanced-json-demo",
    "debug": False,
    "limits": {
        "max_items": 500,
        "timeout_seconds": 2.5,
    },
}

save_json_file(config_path, config)
loaded_config = load_json_file(config_path)

print(config_path.read_text(encoding="utf-8"))
assert loaded_config == config


{
  "app": "advanced-json-demo",
  "debug": false,
  "limits": {
    "max_items": 500,
    "timeout_seconds": 2.5
  }
}



# Problem 20 — Build a recursive JSON-compatibility checker

Before serialization, determine whether a value is directly representable by JSON without custom conversion.

Rules for this exercise:

- keys must be strings
- values may be `None`, `bool`, `int`, finite `float`, `str`, `list`, or `dict`
- tuples are deliberately rejected
- non-finite floats are rejected
- report the exact path of the first failure


## Solution 20


In [48]:

class JSONCompatibilityError(TypeError):
    pass


def assert_direct_json_compatible(value: Any, path: str = "$") -> None:
    if value is None or isinstance(value, (str, bool, int)):
        return

    if isinstance(value, float):
        if not math.isfinite(value):
            raise JSONCompatibilityError(
                f"{path}: non-finite float is not allowed"
            )
        return

    if isinstance(value, list):
        for index, item in enumerate(value):
            assert_direct_json_compatible(item, f"{path}[{index}]")
        return

    if isinstance(value, dict):
        for key, item in value.items():
            if not isinstance(key, str):
                raise JSONCompatibilityError(
                    f"{path}: key {key!r} is not a string"
                )
            assert_direct_json_compatible(item, f"{path}.{key}")
        return

    raise JSONCompatibilityError(
        f"{path}: unsupported type {type(value).__name__}"
    )


In [49]:

good = {
    "name": "demo",
    "values": [1, 2, 3.5, None, True],
}

assert_direct_json_compatible(good)
print("Good payload accepted.")


Good payload accepted.


In [50]:

bad = {
    "name": "demo",
    "nested": {
        "coordinates": (10, 20),
    },
}

try:
    assert_direct_json_compatible(bad)
except JSONCompatibilityError as exc:
    print(exc)


$.nested.coordinates: unsupported type tuple


# Problem 21 — A mini serialization test suite

A robust serializer should be tested for:

- successful round trips
- expected type restoration
- invalid payload rejection
- duplicate-key rejection
- strict non-finite-number rejection
- deterministic output

Write lightweight tests using plain `assert`.


## Solution 21


In [51]:

def test_decimal_round_trip():
    value = Decimal("0.1000000000000000000000001")
    text = json.dumps({"x": value}, default=encode_decimal)
    restored = json.loads(text, object_hook=decode_tagged_object)
    assert restored["x"] == value
    assert isinstance(restored["x"], Decimal)


def test_duplicate_key_rejected():
    text = '{"x": 1, "x": 2}'
    try:
        json.loads(text, object_pairs_hook=reject_duplicate_keys)
    except ValueError:
        return
    raise AssertionError("Duplicate key should have been rejected")


def test_nan_rejected():
    try:
        json.dumps({"x": float("nan")}, allow_nan=False)
    except ValueError:
        return
    raise AssertionError("NaN should have been rejected")


def test_deterministic_output():
    left = {"b": 2, "a": 1}
    right = {"a": 1, "b": 2}
    assert deterministic_json(left) == deterministic_json(right)


def test_order_codec_round_trip():
    text = OrderJSONCodec.dumps(order)
    restored = OrderJSONCodec.loads(text)
    assert restored == order


tests = [
    test_decimal_round_trip,
    test_duplicate_key_rejected,
    test_nan_rejected,
    test_deterministic_output,
    test_order_codec_round_trip,
]

for test in tests:
    test()
    print("PASS", test.__name__)


PASS test_decimal_round_trip
PASS test_duplicate_key_rejected
PASS test_nan_rejected
PASS test_deterministic_output
PASS test_order_codec_round_trip


# Problem 22 — Capstone: event envelope with typed payloads

Design an event envelope for a distributed application.

Requirements:

- `event_id`: UUID
- `event_type`: string
- `occurred_at`: timezone-aware datetime
- `schema_version`: integer
- `payload`: JSON-native document
- deterministic serialization
- duplicate key rejection
- strict validation when decoding


## Solution 22


In [52]:

@dataclass(frozen=True)
class Event:
    event_id: UUID
    event_type: str
    occurred_at: datetime
    schema_version: int
    payload: dict[str, Any]


In [53]:

def event_to_document(event: Event) -> dict[str, Any]:
    if event.occurred_at.tzinfo is None:
        raise ValueError("occurred_at must be timezone-aware")

    if event.schema_version < 1:
        raise ValueError("schema_version must be >= 1")

    assert_direct_json_compatible(event.payload)

    return {
        "event_id": str(event.event_id),
        "event_type": event.event_type,
        "occurred_at": event.occurred_at.isoformat(),
        "schema_version": event.schema_version,
        "payload": event.payload,
    }


def event_from_document(document: dict[str, Any]) -> Event:
    required = {
        "event_id",
        "event_type",
        "occurred_at",
        "schema_version",
        "payload",
    }

    missing = required - document.keys()
    if missing:
        raise ValueError(f"Missing required fields: {sorted(missing)}")

    occurred_at = datetime.fromisoformat(document["occurred_at"])

    if occurred_at.tzinfo is None:
        raise ValueError("occurred_at must be timezone-aware")

    schema_version = document["schema_version"]
    if not isinstance(schema_version, int) or isinstance(schema_version, bool):
        raise ValueError("schema_version must be an integer")

    if schema_version < 1:
        raise ValueError("schema_version must be >= 1")

    payload = document["payload"]
    if not isinstance(payload, dict):
        raise ValueError("payload must be a JSON object")

    assert_direct_json_compatible(payload)

    return Event(
        event_id=UUID(document["event_id"]),
        event_type=document["event_type"],
        occurred_at=occurred_at,
        schema_version=schema_version,
        payload=payload,
    )


In [54]:

def serialize_event(event: Event) -> str:
    return deterministic_json(event_to_document(event))


def deserialize_event(text: str) -> Event:
    document = json.loads(
        text,
        object_pairs_hook=reject_duplicate_keys,
    )
    return event_from_document(document)


In [55]:

event = Event(
    event_id=uuid4(),
    event_type="order.created",
    occurred_at=datetime(2026, 8, 7, 15, 0, tzinfo=timezone.utc),
    schema_version=1,
    payload={
        "order_number": "ORD-1001",
        "item_count": 3,
        "priority": False,
    },
)

event_json = serialize_event(event)
restored_event = deserialize_event(event_json)

print(event_json)
print(restored_event)

assert restored_event == event


{"event_id":"48aad07b-c670-4a12-abb2-e005c84942ca","event_type":"order.created","occurred_at":"2026-08-07T15:00:00+00:00","payload":{"item_count":3,"order_number":"ORD-1001","priority":false},"schema_version":1}
Event(event_id=UUID('48aad07b-c670-4a12-abb2-e005c84942ca'), event_type='order.created', occurred_at=datetime.datetime(2026, 8, 7, 15, 0, tzinfo=datetime.timezone.utc), schema_version=1, payload={'item_count': 3, 'order_number': 'ORD-1001', 'priority': False})


## Capstone extension — Batch events as JSON Lines


In [56]:

event_batch = [
    Event(
        event_id=uuid4(),
        event_type="user.login",
        occurred_at=datetime(2026, 8, 7, 15, 1, tzinfo=timezone.utc),
        schema_version=1,
        payload={"user_id": "u-1"},
    ),
    Event(
        event_id=uuid4(),
        event_type="order.created",
        occurred_at=datetime(2026, 8, 7, 15, 2, tzinfo=timezone.utc),
        schema_version=1,
        payload={"order_id": "o-1", "total": 49.99},
    ),
    Event(
        event_id=uuid4(),
        event_type="user.logout",
        occurred_at=datetime(2026, 8, 7, 15, 3, tzinfo=timezone.utc),
        schema_version=1,
        payload={"user_id": "u-1"},
    ),
]

event_log_path = Path("event_log.jsonl")

with event_log_path.open("w", encoding="utf-8") as f:
    for item in event_batch:
        f.write(serialize_event(item))
        f.write("\n")

print(event_log_path.read_text(encoding="utf-8"))


{"event_id":"208be296-d88c-40a3-a328-7bfee61c0212","event_type":"user.login","occurred_at":"2026-08-07T15:01:00+00:00","payload":{"user_id":"u-1"},"schema_version":1}
{"event_id":"48c44a5f-215d-44a9-bacc-3c87dd7d41cf","event_type":"order.created","occurred_at":"2026-08-07T15:02:00+00:00","payload":{"order_id":"o-1","total":49.99},"schema_version":1}
{"event_id":"71296419-4036-439d-a512-34f95f4b4204","event_type":"user.logout","occurred_at":"2026-08-07T15:03:00+00:00","payload":{"user_id":"u-1"},"schema_version":1}



In [57]:

restored_batch = []

with event_log_path.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        try:
            restored_batch.append(deserialize_event(line))
        except Exception as exc:
            raise ValueError(
                f"Failed to decode event on line {line_number}"
            ) from exc

assert restored_batch == event_batch
print(f"Restored {len(restored_batch)} events.")


Restored 3 events.


# Additional challenge problems

Try these without looking up a ready-made solution first.

### Challenge A
Extend the tagged codec to support `frozenset` while preserving the difference between `set` and `frozenset`.

### Challenge B
Implement a size limit before decoding untrusted JSON text.

### Challenge C
Reject JSON documents nested deeper than a chosen depth after parsing.

### Challenge D
Create a migration pipeline from schema version 1 → 2 → 3.

### Challenge E
Serialize a tree of dataclasses recursively without using `pickle`.

### Challenge F
Write a `loads_decimal()` helper that parses both JSON integers and floats as `Decimal`.

### Challenge G
Create a hash of deterministic JSON using `hashlib.sha256`.

### Challenge H
Design an API response envelope containing:
- request ID
- timestamp
- success flag
- result
- structured error information

### Challenge I
Benchmark compact JSON vs indented JSON for size.

### Challenge J
Write a redaction function that removes keys such as `password`, `token`, and `secret` before logging JSON.


# Reference: compact patterns worth remembering


In [58]:

# Python object -> JSON string
text = json.dumps({"a": 1})

# JSON string -> Python object
obj = json.loads(text)

# Python object -> JSON file
with open("data.json", "w", encoding="utf-8") as f:
    json.dump({"a": 1}, f, indent=2)

# JSON file -> Python object
with open("data.json", "r", encoding="utf-8") as f:
    obj = json.load(f)

# Pretty JSON
pretty = json.dumps({"a": 1}, indent=2)

# Compact JSON
compact = json.dumps({"a": 1}, separators=(",", ":"))

# Stable key ordering
stable = json.dumps({"b": 2, "a": 1}, sort_keys=True)

# Human-readable Unicode
unicode_json = json.dumps({"hello": "世界"}, ensure_ascii=False)

# Strict finite-number behavior
strict = json.dumps({"x": 1.5}, allow_nan=False)

# Parse JSON decimal numbers as Decimal
precise = json.loads('{"x": 0.1}', parse_float=Decimal)

# Custom serialization fallback
custom = json.dumps(
    {"x": Decimal("1.25")},
    default=lambda obj: str(obj) if isinstance(obj, Decimal) else None,
)


# Final takeaways

- JSON is simple because it has a deliberately small type system.
- Serialization is not just "turn an object into a string"; it is a **representation design problem**.
- Round-trip equality is not guaranteed unless you define how lost type information is represented.
- Tagged objects are useful when type restoration matters.
- For long-lived systems, use explicit schema versions and migration logic.
- For financial precision, prefer `Decimal` and a deliberate representation.
- For interoperability, consider `allow_nan=False`.
- For security-sensitive documents, reject duplicate keys and validate the decoded structure.
- For large record streams, JSON Lines is often simpler and more memory-efficient than one giant array.
- For cryptographic or cache keys, deterministic JSON must follow a clearly defined canonicalization rule.
